# Colab Full Control MCP Setup

This notebook mounts Drive, clones or updates the repository, installs dependencies, starts the MCP server, opens a Cloudflare Tunnel, and prints the Codex MCP config snippet.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/CopyyQ/colab-full-control-mcp.git'
PROJECT_DIR = Path('/content/colab-full-control-mcp')
SRC_DIR = PROJECT_DIR / 'src'

os.chdir('/content')
if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
    print('Updated existing checkout:', PROJECT_DIR)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
    print('Cloned repository into:', PROJECT_DIR)
os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())

In [ ]:
%pip install -r requirements.txt
%pip install -e .

In [ ]:
import os
import sys
from getpass import getpass

src_dir = str(SRC_DIR)
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
existing_pythonpath = os.environ.get('PYTHONPATH', '')
os.environ['PYTHONPATH'] = src_dir if not existing_pythonpath else src_dir + ':' + existing_pythonpath

os.environ['COLAB_MCP_TOKEN'] = getpass('COLAB_MCP_TOKEN: ')
os.environ['PERMISSION_PROFILE'] = 'DEVELOPER'
os.environ['ALLOWED_ROOTS'] = '/content,/content/drive/MyDrive'
os.environ['UNRESTRICTED_RUNTIME_MODE'] = 'false'
print('Kernel src path ready:', src_dir)

In [ ]:
import subprocess
import sys
import time

subprocess.run([sys.executable, 'scripts/stop_server.py'], cwd=str(PROJECT_DIR), check=False, capture_output=True, text=True)
server_proc = subprocess.Popen([sys.executable, 'scripts/start_server.py'], cwd=str(PROJECT_DIR))
time.sleep(2)
print('server pid =', server_proc.pid)

In [ ]:
import subprocess
import sys
import time

last_output = ''
for attempt in range(10):
    result = subprocess.run([sys.executable, 'scripts/health_check.py'], cwd=str(PROJECT_DIR), capture_output=True, text=True)
    last_output = (result.stdout or result.stderr).strip()
    if result.returncode == 0:
        print(last_output)
        break
    time.sleep(1)
else:
    raise RuntimeError(last_output or 'Health check failed')

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

if shutil.which('cloudflared') is None:
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], check=True)
    subprocess.run(['dpkg', '-i', 'cloudflared-linux-amd64.deb'], check=False)
    subprocess.run(['apt-get', '-f', 'install', '-y'], check=True)
else:
    print('cloudflared already installed')

subprocess.run([sys.executable, 'scripts/stop_tunnel.py'], cwd=str(PROJECT_DIR), check=False, capture_output=True, text=True)
result = subprocess.run([sys.executable, 'scripts/start_tunnel.py', '--server-url', 'http://127.0.0.1:8000'], cwd=str(PROJECT_DIR), check=True, capture_output=True, text=True)
print(result.stdout.strip())

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

state = json.loads(Path('/content/.colab_full_control_mcp/jobs/cloudflared_state.json').read_text())
public_url = state['url'] + '/mcp'
print('Public MCP URL:', public_url)
subprocess.run([sys.executable, 'scripts/print_codex_config.py', '--url', public_url], cwd=str(PROJECT_DIR), check=True)

In [ ]:
import sys

src_dir = str(SRC_DIR)
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from colab_full_control_mcp.tools import TOOL_REGISTRY

print('Registered tools:', sum(len(names) for names in TOOL_REGISTRY.values()))
print('Import path:', src_dir)

In [ ]:
import subprocess
import sys

def summarize_stop(label: str, script_name: str) -> None:
    result = subprocess.run([sys.executable, script_name], cwd=str(PROJECT_DIR), check=False, capture_output=True, text=True)
    combined = '\n'.join(part.strip() for part in (result.stdout, result.stderr) if part and part.strip())
    if result.returncode != 0:
        raise RuntimeError(f'{label} stop failed: {combined or result.returncode}')
    if 'Stopped cloudflared pid=' in combined or 'Sent termination signal to MCP server pid=' in combined:
        summary = 'stop signal sent'
    elif 'was already stopped or unavailable' in combined or 'Nothing to stop.' in combined:
        summary = 'already stopped; stale state cleaned up if needed'
    else:
        summary = combined or 'completed'
    print(f'{label}: {summary}')

summarize_stop('Tunnel', 'scripts/stop_tunnel.py')
summarize_stop('MCP server', 'scripts/stop_server.py')
print('Stop sequence completed.')